# 關鍵點偵測模型訓練

## 載入資料

In [ ]:
import pandas as pd
import os
import cv2
import matplotlib.pyplot as plt
import numpy as np
from shared.functions import *

image_data_dir = "cloth_data_gen/output/images"
keypoint_data_dir = "cloth_data_gen/output/keypoints"

img_arr = []
keypoints_img_arr = []
for img_file in os.listdir(image_data_dir):
    if img_file.endswith('.png'):
        name = img_file.split('.')[0]
        keypoint_file = os.path.join(keypoint_data_dir, name + '.txt')
        image_path = os.path.join(image_data_dir, img_file)
        img = cv2.imread(image_path)
        keypoints = pd.read_csv(keypoint_file)
        pixels_coords = keypoints[['x_pixel', 'y_pixel']].values
        kimg = np.zeros((img.shape[0], img.shape[1]), dtype=np.uint8)
        karr = []
        # check if all pixels coordinates are within the image bounds
        if pixels_coords.shape[0] > 0 and np.all((pixels_coords[:, 0] >= 0) & (pixels_coords[:, 0] < img.shape[1]) &
                                                  (pixels_coords[:, 1] >= 0) & (pixels_coords[:, 1] < img.shape[0])):
            kp_img = np.zeros((128, 128))
            for point in pixels_coords:
                kp_img[int(point[1]), int(point[0])] = 1
            keypoints_img_arr.append(kp_img)
            img_arr.append(img)
img_arr = np.array(img_arr)
keypoints_img_arr = np.array(keypoints_img_arr)

In [ ]:
import torch
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
import numpy as np
import torchvision.transforms.functional as TF
import random

class KeypointDataset(Dataset):
    def __init__(self, images, keypoints, transform=None):
        self.images = images.astype(np.float32) / 255
        self.keypoints = keypoints.astype(np.float32)
        self.transform = transform

    def __len__(self):
        return len(self.images)

    def __getitem__(self, idx):
        img = self.images[idx]  # shape (400, 400, 3)
        kp = self.keypoints[idx]  # shape (4, 2)
        img = np.transpose(img, (2, 0, 1))  # channels first
        sample = {'image': torch.from_numpy(img), 'keypoints': torch.from_numpy(kp)}
        if self.transform:
            sample = self.transform(sample)
        return sample

import torch
import torchvision.transforms.functional as TF
import random

class RandomRotateFlip:
    """
    Randomly applies:
    - A rotation by any angle in [0, 360)
    - Optionally, a horizontal flip with 50% chance after rotation
    """
    def __call__(self, sample):
        image, keypoints = sample['image'], sample['keypoints']
        # image: (C, H, W)
        # keypoints: (N, H, W) or (H, W)

        # --- Random rotation ---
        angle = random.uniform(0, 360)
        image = TF.rotate(image, angle, interpolation=TF.InterpolationMode.BILINEAR)
        # For keypoints as heatmaps, use same rotate (assume keypoints is Tensor [N,H,W] or [H,W])
        # If N, treat each as a channel
        if keypoints.ndim == 3:
            keypoints = TF.rotate(keypoints, angle, interpolation=TF.InterpolationMode.BILINEAR)
        else:
            keypoints = TF.rotate(keypoints.unsqueeze(0), angle, interpolation=TF.InterpolationMode.BILINEAR).squeeze(0)

        # --- Random flip after rotation ---
        if random.random() < 0.5:
            image = TF.hflip(image)
            keypoints = TF.hflip(keypoints)
        if random.random() < 0.5:
            image = TF.vflip(image)
            keypoints = TF.vflip(keypoints)

        return {'image': image, 'keypoints': keypoints}

In [ ]:
rotate_transform = RandomRotateFlip()

# Create the full dataset without transform
full_dataset = KeypointDataset(img_arr, keypoints_img_arr, transform=None)

# Split indices for train/test
train_size = int(0.8 * len(full_dataset))
test_size = len(full_dataset) - train_size
train_indices, test_indices = torch.utils.data.random_split(range(len(full_dataset)), [train_size, test_size])

# Create train and test datasets with/without transform
train_dataset = torch.utils.data.Subset(KeypointDataset(img_arr, keypoints_img_arr, transform=rotate_transform), train_indices)
test_dataset = torch.utils.data.Subset(KeypointDataset(img_arr, keypoints_img_arr, transform=None), test_indices)

trainloader = DataLoader(train_dataset, batch_size=8, shuffle=True, pin_memory=True)
testloader = DataLoader(test_dataset, batch_size=8, shuffle=False, pin_memory=True)

## 測試關鍵點跟圖片的正確性

In [ ]:
pair = full_dataset.__getitem__(9)
img = pair["image"].numpy().copy()
img = np.transpose(img, (1, 2, 0))
img = cv2.cvtColor(img, cv2.COLOR_RGB2BGR)

kp = pair["keypoints"].numpy()
print(kp.shape)
for i in range(kp.shape[0]):
    for j in range(kp.shape[1]):
        if kp[i,j] > 0.1:
            cv2.circle(img, (j, i), 1, (0,0,255), -1)
plt.imshow(img)
plt.show()

## 訓練迴圈

In [ ]:
load_model = False

In [ ]:
from models.utils import *

In [ ]:
def spatial_klloss(pred_map, target_map, eps=1e-8):
    # pred_map: after spatial softmax, (B, 1, H, W)
    # target_map: one-hot or few-hot, (B, H, W)
    B, _, H, W = pred_map.shape
    pred = pred_map.view(B, -1) + eps  # avoid log(0)
    target = target_map.view(B, -1) + eps
    pred_log = pred.log()
    target = target / target.sum(dim=1, keepdim=True)  # ensure sum-to-1; safe for multi-keypoint
    return (target * (target.log() - pred_log)).sum(dim=1).mean()

def kl_heatmap_loss(pred_hm, gt_hm, mask=None, reduction='mean'):
    """
    pred_hm: (B, 1, H, W) tensor, model output (must be positive, not all zeros)
    gt_hm:   (B, 1, H, W) tensor, ground-truth (should be positive, not all zeros)
    mask:    (B, 1, H, W) optional mask (1=valid, 0=ignored) or None
    reduction: 'mean', 'sum', or 'none'
    Returns: scalar loss
    """
    B, _, H, W = pred_hm.shape
    # Flatten
    pred_flat = pred_hm.view(B, -1)
    gt_flat = gt_hm.view(B, -1)

    # Force positive and normalize to sum=1 for both (prob dists)
    pred_probs = pred_flat.clamp(min=1e-8)
    pred_probs = pred_probs / pred_probs.sum(dim=1, keepdim=True)
    gt_probs = gt_flat.clamp(min=1e-8)
    gt_probs = gt_probs / gt_probs.sum(dim=1, keepdim=True)

    kl_div = F.kl_div(pred_probs.log(), gt_probs, reduction='none').sum(dim=1)  # KL per sample

    if reduction == 'mean':
        return kl_div.mean()
    elif reduction == 'sum':
        return kl_div.sum()
    else:
        return kl_div  # shape (B,)

def batch_gaussian_blur(x, kernel_size=5, sigma=2.0):
    """
    Apply Gaussian blur to a batch of heatmaps and normalize each so the max is 1.
    Args:
        x: Tensor [B, H, W] or [B, 1, H, W]
    Returns:
        Tensor with same shape, blurred and with peak 1 per sample
    """
    unsqueeze = False
    if x.dim() == 3:  # [B, H, W]
        x = x.unsqueeze(1)
        unsqueeze = True
    
    blurred = TF.gaussian_blur(x, kernel_size=[kernel_size, kernel_size], sigma=[sigma, sigma])
    max_vals = blurred.amax(dim=[2, 3], keepdim=True)
    max_vals[max_vals == 0] = 1.0  # Avoid division by zero
    normalized = blurred / max_vals

    if unsqueeze:
        normalized = normalized.squeeze(1)
    return normalized

def batch_entropy(pred_heatmaps):
    """
    pred_heatmaps: [B, C, H, W]
    Returns: [B] entropy per image
    """
    # Flatten spatial dimensions (and optionally channels) for softmax
    B, C, H, W = pred_heatmaps.shape
    flat = pred_heatmaps.view(B, -1)                # [B, C*H*W]
    probs = torch.softmax(flat, dim=1)              # normalize to sum=1 per image
    entropy = -(probs * torch.log(probs + 1e-8)).sum(dim=1)  # [B]
    return entropy

In [ ]:
import torch.optim as optim
import torch.nn as nn
from models.unet import UNet
# from models.yolo_cnn import EnhancedYoloKeypointNet
from models.yolo_vit import HybridKeypointNet
from ultralytics import YOLO
import time

# optimization
from torch.amp import autocast, GradScaler
torch.backends.cudnn.benchmark = True
scaler = GradScaler()

# create device
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

# yolo cnn
# yolo11 = YOLO('yolo11l-seg.pt')  # Or yolo11m-seg.pt, yolo11x-seg.pt, etc.
# backbone_seq = yolo11.model.model[:12]
# backbone = YoloBackbone(backbone_seq, selected_indices=[0,1,2,3,4,5,6,7,8,9,10,11])
# input_dummy = torch.randn(1, 3, 512, 512)
# with torch.no_grad():
#     feats = backbone(input_dummy)
# print("Feature shapes:", [f.shape for f in feats])
# in_channels_list = [f.shape[1] for f in feats]

# device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
# keypoint_net = EnhancedYoloKeypointNet(backbone, in_channels_list)
# model = keypoint_net
# for param in model.backbone.parameters():
#     param.requires_grad = False
# model = model.to(device)

# yolo vit
yolo_model = YOLO('yolov8l.pt')
backbone_seq = yolo_model.model.model[:10]
backbone = YoloBackbone(backbone_seq, selected_indices=[0,1,2,3,4,5,6,7,8,9])
input_dummy = torch.randn(1, 3, 128, 128)
with torch.no_grad():
    feats = backbone(input_dummy)
in_channels_list = [f.shape[1] for f in feats]
keypoint_net = HybridKeypointNet(backbone, in_channels_list)
model = keypoint_net
for param in model.backbone.parameters():
    param.requires_grad = False
# for param in model.diffusion.vit.parameters():
#     param.requires_grad = False
model = model.to(device)

# unet
# model = UNet(in_channels=3, out_channels=4).to(device)

compiled_model = torch.compile(model)
if not load_model:
    optimizer = optim.AdamW(compiled_model.parameters(), lr=1e-5)

    for epoch in range(300):
        time_start = time.time()
        compiled_model.train()
        running_loss = 0.0

        for batch in trainloader:
            images = batch["image"].to(device)
            keypoints = batch["keypoints"].to(device)
            optimizer.zero_grad()

            with autocast("cuda", dtype=torch.float16):      # AMP context, not forcing .half()
                outputs = compiled_model(images)
                keypoints_blur = batch_gaussian_blur(keypoints, kernel_size=31, sigma=3)
                
                # active learning: Uncertainty Sampling using entropy as the uncertainty metric
                entropies = batch_entropy(outputs)
                k = images.size(0) // 2
                topk_vals, topk_idx = torch.topk(entropies, k, largest=True)  # highest entropy first
                selected_outputs = outputs[topk_idx]
                selected_keypoints_blur = keypoints_blur[topk_idx]

                # calculate loss
                loss = kl_heatmap_loss(selected_outputs, selected_keypoints_blur.unsqueeze(1))

            scaler.scale(loss).backward()
            scaler.step(optimizer)
            scaler.update()
            running_loss += loss.item() * images.size(0)
        print(f'Epoch {epoch+1}: Loss {running_loss / len(train_dataset):.4f} time seconds:, {time.time() - time_start}')

    # save the model
    torch.save(compiled_model.state_dict(), 'models/keypoint_model_vit.pth')
else:

    compiled_model.load_state_dict(torch.load('models/keypoint_model_vit.pth', map_location=device))
    compiled_model.eval()

## 模型結果分析

In [ ]:
# Evaluate on the validation set
compiled_model.eval()
val_loss = 0.0
iter = 0
with torch.no_grad():
    for batch in testloader:
        images = batch['image'].to(device)
        keypoints = batch['keypoints'].to(device)
        with autocast("cuda", dtype=torch.float16):
            outputs = compiled_model(images)
            keypoints_blur = batch_gaussian_blur(keypoints, kernel_size=31, sigma=3)
            loss = kl_heatmap_loss(outputs, keypoints_blur.unsqueeze(1))

        # render the predicted keypoints on the image
        for img, kp in zip(images.cpu().numpy(), outputs.cpu().numpy()):
            img = np.transpose(img, (1, 2, 0)) * 255
            # Convert RGB to BGR for OpenCV
            img = cv2.cvtColor(img, cv2.COLOR_RGB2BGR)
            kp = kp[0,:,:]
            peaks = thresholded_locations(kp, 0.003)
            for p in peaks:
                i,j = p
                cv2.circle(img, (int(j), int(i)), 3, (255,0,0), -1)
            img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
            # 撰寫測試集結果於 output 資料集
            cv2.imwrite(f'results/keypoints_{iter}.png', img)
            iter += 1
        val_loss += loss.item() * images.size(0)
    print(f'Validation Loss: {val_loss / len(test_dataset):.4f}')


## 模型視覺化

In [ ]:
from torchview import draw_graph

# Suppose `model` is your nn.Module, and x is a sample input tensor
model_graph = draw_graph(model, input_data=torch.randn((8,3,128,128)), expand_nested=True)
model_graph.visual_graph.render(filename='architecture_full', format='png')
# or to view inline in a Jupyter notebook:
display(model_graph.visual_graph)